# Malaga — Reviews EDA & Sentiment Analysis
**Reviews :** `../../Data/raw/Malaga/reviews.csv.gz`  
**Listings :** `../../Data/interim/malaga_listings_clean.parquet` (pre-cleaned, for merging)  
**Prerequisite:** run `01_cleaning.ipynb` first.

Covers:
1. Reviews dataset overview (volume, length, temporal trends)
2. Text preprocessing
3. Language detection & translation (optional — cached)
4. TextBlob sentiment scoring
5. Sentiment analysis across dimensions (neighbourhood, property type, price, superhost)
6. Word clouds

In [ ]:
# !pip install textblob langdetect deep-translator tqdm wordcloud  # run once if needed

In [ ]:
import sys, re, os, time
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from wordcloud import WordCloud
from textblob import TextBlob
from tqdm import tqdm
tqdm.pandas()

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 300)

## 1 · Load data

### 1.1 Reviews

In [ ]:
df_reviews = pd.read_csv(
    "../../Data/raw/Malaga/reviews.csv.gz",
    compression="gzip",
    nrows=100_000,      # increase or remove for full dataset
    low_memory=False,
)
df_reviews["date"] = pd.to_datetime(df_reviews["date"], errors="coerce")
print(f"Reviews loaded: {len(df_reviews):,}")
print(f"Columns: {df_reviews.columns.tolist()}")
df_reviews.head()

### 1.2 Cleaned listings (for merging)

In [ ]:
listings_meta_cols = [
    "id", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "property_type_std", "room_type", "price", "price_cat",
    "host_is_superhost", "review_scores_rating", "host_tenure_years",
]
l_df = pd.read_parquet("../../Data/interim/malaga_listings_clean.parquet")
available = [c for c in listings_meta_cols if c in l_df.columns]
l_meta = l_df[available].copy()
print(f"Listing metadata: {l_meta.shape}")

## 2 · Reviews dataset overview

In [ ]:
# Shape, types, missing
df_reviews.info()
print("\nMissing values:")
print(df_reviews.isna().mean().sort_values(ascending=False))

In [ ]:
# Review length (chars and words)
df_reviews["review_len_chars"] = df_reviews["comments"].str.len()
df_reviews["review_len_words"] = df_reviews["comments"].str.split().str.len()

display(df_reviews[["review_len_chars", "review_len_words"]].describe())

fig = px.histogram(
    df_reviews, x="review_len_words", nbins=60,
    title="Distribution of review length (words)",
    labels={"review_len_words": "Word count"},
)
fig.show()

In [ ]:
# Volume over time
monthly_vol = (
    df_reviews.set_index("date")
    .resample("ME")
    .size()
    .reset_index(name="review_count")
)
fig = px.line(monthly_vol, x="date", y="review_count",
              title="Monthly review volume")
fig.show()

In [ ]:
# Top listings by review count
top_listings = (
    df_reviews.groupby("listing_id")["id"]
    .count()
    .sort_values(ascending=False)
    .head(15)
    .reset_index(name="review_count")
)
display(top_listings)

In [ ]:
# Reviewers — how many listings does a reviewer tend to review?
reviewer_activity = (
    df_reviews.groupby("reviewer_id")["listing_id"]
    .nunique()
    .reset_index(name="listings_reviewed")
)
display(reviewer_activity["listings_reviewed"].describe())

fig = px.histogram(
    reviewer_activity, x="listings_reviewed", nbins=30,
    title="How many different listings each reviewer reviews",
    labels={"listings_reviewed": "Distinct listings reviewed"},
)
fig.show()

## 3 · Text preprocessing

In [ ]:
def clean_comment(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    return re.sub(r"\s+", " ", text).strip()

df_pp = df_reviews.dropna(subset=["comments"]).copy()
df_pp["review_len_words"] = df_pp["comments"].str.split().str.len()
df_pp = df_pp[df_pp["review_len_words"] >= 3].copy()
df_pp["clean_comment"] = df_pp["comments"].apply(clean_comment)

print(f"Reviews after preprocessing: {len(df_pp):,}")
print(f"  Dropped (missing / too short): {len(df_reviews) - len(df_pp):,}")
df_pp[["comments", "clean_comment"]].head(3)

## 4 · Language detection & translation (optional)

Translation via Google Translate (`deep-translator`) takes several hours on a
large dataset.  The result is cached to a local CSV so subsequent runs are
instant.  If the libraries are not installed the notebook skips this step and
runs sentiment directly on the cleaned text.

In [ ]:
CACHE_CSV = "translated_reviews_malaga.csv"

try:
    from langdetect import detect, LangDetectException
    from deep_translator import GoogleTranslator
    HAS_TRANSLATE = True
except ImportError:
    HAS_TRANSLATE = False
    print("langdetect / deep-translator not installed — skipping translation.")

if HAS_TRANSLATE and not os.path.exists(CACHE_CSV):
    def detect_lang(t):
        try:
            return detect(t) if isinstance(t, str) and len(t) > 10 else "unknown"
        except LangDetectException:
            return "unknown"

    print("Detecting languages…")
    df_pp["detected_language"] = df_pp["clean_comment"].progress_apply(detect_lang)

    # ── Language distribution chart ──
    lang_counts = df_pp["detected_language"].value_counts()
    fig = px.bar(lang_counts.head(15).reset_index(),
                 x="detected_language", y="count",
                 title="Top detected languages")
    fig.show()

    non_en = (
        (df_pp["detected_language"] != "en") &
        (df_pp["detected_language"] != "unknown")
    )
    df_pp["translated_comment"] = df_pp["clean_comment"]

    if non_en.sum() > 0:
        print(f"Translating {non_en.sum():,} non-English reviews…")
        idx_list, translated = df_pp[non_en].index.tolist(), []
        for i, idx in enumerate(tqdm(idx_list)):
            try:
                t = GoogleTranslator(source="auto", target="en").translate(
                    df_pp.loc[idx, "clean_comment"])
                translated.append(t or df_pp.loc[idx, "clean_comment"])
            except Exception:
                translated.append(df_pp.loc[idx, "clean_comment"])
            if i % 50 == 0 and i > 0:
                time.sleep(0.5)
        df_pp.loc[non_en, "translated_comment"] = translated

    df_pp.to_csv(CACHE_CSV, index=False)
    print(f"Translation cache saved → {CACHE_CSV}")

elif os.path.exists(CACHE_CSV):
    df_pp = pd.read_csv(CACHE_CSV)
    df_pp["date"] = pd.to_datetime(df_pp["date"], errors="coerce")
    print(f"Loaded translation cache: {len(df_pp):,} reviews")

df_pp["final_text"] = df_pp.get("translated_comment", df_pp["clean_comment"])

## 5 · Sentiment analysis

In [ ]:
POS_THRESH =  0.1
NEG_THRESH = -0.1

def get_sentiment(text):
    if not isinstance(text, str):
        return (0.0, 0.0)
    s = TextBlob(text).sentiment
    return (s.polarity, s.subjectivity)

print("Scoring sentiment…")
df_pp["polarity"], df_pp["subjectivity"] = zip(
    *df_pp["final_text"].progress_apply(get_sentiment)
)

def label(score):
    if score >= POS_THRESH: return "Positive"
    if score <= NEG_THRESH: return "Negative"
    return "Neutral"

df_pp["sentiment_label"] = df_pp["polarity"].apply(label)
dist = df_pp["sentiment_label"].value_counts()
print(dist.to_string())
print(f"\nMean polarity: {df_pp['polarity'].mean():.3f}")

fig = px.pie(
    df_pp, names="sentiment_label",
    title="Overall sentiment distribution",
    color="sentiment_label",
    color_discrete_map={"Positive": "#2ecc71", "Neutral": "#f1c40f", "Negative": "#e74c3c"},
)
fig.show()

### 5.1 Sentiment over time

In [ ]:
df_pp["year_month"] = df_pp["date"].dt.to_period("M")

monthly_sentiment = (
    df_pp.groupby("year_month")
    .agg(mean_polarity=("polarity", "mean"), count=("polarity", "count"))
)

fig = px.line(
    monthly_sentiment.reset_index()
        .assign(year_month=lambda x: x["year_month"].astype(str)),
    x="year_month", y="mean_polarity",
    title="Average sentiment polarity over time",
    labels={"mean_polarity": "Mean polarity", "year_month": "Month"},
)
fig.show()

# Stacked sentiment proportion by month
monthly_prop = (
    df_pp.groupby(["year_month", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
    .apply(lambda x: x / x.sum(), axis=1)
)
fig2 = px.area(
    monthly_prop.reset_index()
        .assign(year_month=lambda x: x["year_month"].astype(str))
        .melt(id_vars="year_month"),
    x="year_month", y="value", color="variable",
    title="Sentiment proportion over time (stacked area)",
    color_discrete_map={"Positive":"#2ecc71","Neutral":"#f1c40f","Negative":"#e74c3c"},
)
fig2.show()

### 5.2 Sentiment by listing dimension

In [ ]:
# Merge with listing metadata
df_full = df_pp.merge(l_meta, left_on="listing_id", right_on="id", how="left")

def sentiment_bars(df, dim):
    if dim not in df.columns:
        print(f"Column '{dim}' not available — skipping.")
        return
    grp = (
        df.groupby(dim, observed=True)
        .agg(
            positive=("sentiment_label", lambda x: (x == "Positive").mean()),
            negative=("sentiment_label", lambda x: (x == "Negative").mean()),
            count   =("sentiment_label", "count"),
        )
        .query("count >= 30")
        .sort_values("positive", ascending=False)
        .reset_index()
    )
    fig = px.bar(
        grp, x=dim, y=["positive", "negative"],
        title=f"Sentiment proportions by {dim}",
        barmode="group",
        color_discrete_map={"positive": "#2ecc71", "negative": "#e74c3c"},
    )
    fig.update_layout(xaxis_tickangle=-40)
    fig.show()

for dim in [
    "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "property_type_std", "price_cat", "host_is_superhost",
]:
    sentiment_bars(df_full, dim)

### 5.3 Sentiment vs numerical review score

In [ ]:
if "review_scores_rating" in df_full.columns:
    df_rating = df_full.dropna(subset=["review_scores_rating"]).copy()
    df_rating["rating_bucket"] = pd.cut(
        df_rating["review_scores_rating"],
        bins=[1, 3, 4, 4.5, 5], labels=["1–3", "3–4", "4–4.5", "4.5–5"],
        include_lowest=True,
    )
    rating_sent = (
        df_rating.groupby("rating_bucket", observed=True)
        .agg(avg_polarity=("polarity","mean"), count=("polarity","count"))
        .reset_index()
    )
    display(rating_sent)
    corr = df_rating["polarity"].corr(df_rating["review_scores_rating"])
    print(f"Polarity ↔ rating Pearson r = {corr:.3f}")

    fig = px.line(
        rating_sent, x="rating_bucket", y="avg_polarity", markers=True,
        title="Average polarity by star-rating bucket",
    )
    fig.show()

## 6 · Word clouds

In [ ]:
for label_str, cmap in [("Positive", "Greens"), ("Negative", "Reds")]:
    subset = df_pp[df_pp["sentiment_label"] == label_str]["clean_comment"].astype(str)
    if len(subset) == 0:
        print(f"No {label_str} reviews to plot.")
        continue
    wc = WordCloud(
        width=1200, height=600, background_color="white",
        max_words=150, colormap=cmap,
    ).generate(" ".join(subset))
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word cloud — {label_str} reviews", fontsize=18)
    plt.tight_layout()
    plt.show()

## 7 · Key findings

_Fill in the most important insights from the reviews analysis._